In [1]:
from pypdf import PdfReader
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
import numpy as np
import json
from rapidfuzz import process, fuzz
import random
from label_studio_sdk import LabelStudio

In [ ]:
source ~/.zshrc

UsageError: Line magic function `%source` not found.


In [21]:
pattern_date = r"\b\d{1,2}\s+(?:janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)\s+\d{4}\b"
pattern_date_paru = r"\b\d{1,2}\s+(?:janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)\s+\d{4}\s*-"

In [4]:
SOURCE_DIR = "pdf_print"
OUTPUT_DIR = "json_output"

In [23]:
def extract_text_from_pdf(reader):
    txt = []
    for page in reader.pages:
        text = page.extract_text(extraction_mode="layout", layout_mode_space_vertically=False, layout_mode_space_horizontally=False)
        text = re.sub(r" +", " ", text)
        text = re.sub(r"\t+", "", text)
        txt.append(text)

    txt = "\n".join(txt)

    return txt

In [123]:
def normaliser_journal(j):
    return re.sub(
        r',?\s*(no\.?|n°)\s*(Vol\s*[:.]\s*\d+(?:\s*No\s*[:.]\s*\d+)?|\d+)',
        '',
        j,
        flags=re.IGNORECASE
    ).strip()

In [163]:
def extract_metadata(article):
    journaux = []
    dates = []

    lines = article.split("\n")
    nb = len(lines)
    if nb < 2:
        return

    journaux.append(normaliser_journal(lines[0]))

    # On vérifie si la deuxième ligne contient la date de la publication de l'article
    n = 1
    match = re.search(pattern_date, lines[n], re.IGNORECASE)

    # Sinon on continue à chercher la date dans les lignes suivantes jusqu'à ce qu'on trouve une correspondance ou qu'on atteigne la fin de l'article
    while n < nb - 1 and not match:
        n += 1
        match = re.search(pattern_date, lines[n], re.IGNORECASE)

    # On ajoute la date trouvée à la liste des dates
    if match:
        date = match.group(0)
        dates.append(date)

    n += 1

    aussi_paru = (
        n < nb and re.search("Aussi paru dans", lines[n])
    ) or (
        n + 1 < nb and re.search("Aussi paru dans", lines[n + 1])
    )

    if aussi_paru:
        match = re.search(pattern_date, lines[n], re.IGNORECASE)
        if match:
            date = match.group(0)
            dates.append(date)

            journal = re.split(r'\s*-', lines[n], maxsplit=1)[1]
            journaux.append(normaliser_journal(journal))

        n += 1
        while n < nb and re.search(pattern_date_paru, lines[n], re.IGNORECASE):
            date = re.search(pattern_date, lines[n], re.IGNORECASE).group(0)
            dates.append(date)
            journal = re.split(r'\s*-', lines[n], maxsplit=1)[1]
            journaux.append(normaliser_journal(journal))
            n += 1


    texte_article = "\n".join(lines[n:])

    metadata = {
        "journaux": journaux,
        "dates": dates,
        "texte": texte_article
    }

    return(metadata)

In [159]:
def split_into_articles(path):
    reader = PdfReader(path)
    txt = extract_text_from_pdf(reader)
    articles = re.split(r"\bnews.*\n", txt)
    jsons = []
    for article in articles:
        article = article.split('©')
        if len(article) > 1:
            article = '©'.join(article[:-1])
        else:
            article = articles[0]
        metadata = extract_metadata(article)
        if metadata:
            jsons.append(metadata)
    return jsons

In [27]:
def dfs(noeud, graphe, visite, composante): #https://chat.mistral.ai/chat/d52a01f2-0186-4d4b-b46a-279ad43bdd3f
    pile = [noeud]
    while pile:
        n = pile.pop()
        if n not in visite:
            visite.add(n)
            composante.append(n)
            for voisin in graphe[n]:
                if voisin not in visite:
                    pile.append(voisin)

In [28]:
def trouver_composantes_connexes(paires):
    # Créer un graphe à partir des paires
    graphe = defaultdict(list)
    for a, b in paires:
        graphe[a].append(b)
        graphe[b].append(a)

    visites = set()
    composantes_connexes = []

    # Parcourir tous les noeuds du graphe
    for noeud in graphe:
        if noeud not in visites:
            composante = []
            dfs(noeud, graphe, visites, composante)
            composantes_connexes.append(sorted(composante))

    return composantes_connexes

In [29]:
def nettoyer_texte(texte):
    # Supprimer les balises HTML
    texte = re.sub(r'<[^>]+>', ' ', texte)
    # Minuscules
    texte = texte.lower()
    # Supprimer la ponctuation et les caractères spéciaux
    texte = re.sub(r'[^\w\s]', ' ', texte)
    # Normaliser les espaces
    texte = re.sub(r'\s+', ' ', texte).strip()
    return texte

In [30]:
def fusion_jsons(articles):
    journaux = []
    dates = []
    for article in articles:
        for journal, date in zip(article["journaux"], article["dates"]):
            if journal not in journaux:
                journaux.append(journal)
                dates.append(date)
    metadata = {
        "journaux": journaux,
        "dates": dates,
        "texte": articles[0]['texte']
    }

    return(metadata)


In [31]:
def merge_double(articles):
    textes = [nettoyer_texte(a["texte"]) for a in articles]
    try:
        vect = TfidfVectorizer(min_df=1)
        tfidf = vect.fit_transform(textes)
    except ValueError as e:
        print(f"Erreur TF-IDF : {e}")
        return articles
    pairwise_similarity = tfidf * tfidf.T
    arr = pairwise_similarity.toarray()
    np.fill_diagonal(arr, 0)
    index_doublon  = np.transpose((arr>=0.90).nonzero())
    groupe_doublon = trouver_composantes_connexes(index_doublon) #regroupe les doublons par groupe en regardant quels binômes sont transitifs
    # Indices à fusionner
    indices_fusionnes = {i for groupe in groupe_doublon for i in groupe}

    resultat = []
    # Ajouter les articles non dupliqués tels quels
    for i, article in enumerate(articles):
        if i not in indices_fusionnes:
            resultat.append(article)

    # Fusionner les groupes
    for groupe in groupe_doublon:
        resultat.append(fusion_jsons([articles[i] for i in groupe]))

    return resultat

In [32]:
def nom_fichier(article, index=None):
    """
    Génère un nom de fichier à partir des métadonnées du JSON.
    Exemple : le-soleil_12-mars-2024_0042.json
    """
    # Premier journal
    journal = article["journaux"][0] if article["journaux"] else "inconnu"
    journal = re.sub(r'\s*\([^)]*\)', '', journal)  # supprime les parenthèses
    journal = journal.strip().lower()
    journal = re.sub(r'[^a-z0-9]+', '-', journal)   # slugify

    # Date
    date = article["dates"][0] if article["dates"] else "sans-date"
    date = date.strip().lower()
    date = re.sub(r'\s+', '-', date)
    date = re.sub(r'[^a-z0-9\-]', '', date)

    # Suffixe index pour éviter les collisions
    suffixe = f"_{index:04d}" if index is not None else ""

    return f"{journal}_{date}{suffixe}.json"

In [33]:
def normalise_nom(string):
    if re.search(r", Le\b", string):
        string = re.sub(r", Le\b", "", string)
        string = "Le " + string
    elif re.search(r", La\b", string):
        string = re.sub(r", La\b", "", string)
        string = "La " + string
    elif re.search(r", L'", string):
        string = re.sub(r", L'", "", string)
        string = "L'" + string
    elif re.search(r", Les\b", string):
        string = re.sub(r", Les\b", "", string)
        string = "Les " + string
    return string.strip()

In [ ]:
with open("sources.txt", "r", encoding="utf-8") as f: #création du json d'origine des sources
    sources = f.read().splitlines()

failures = []
liste_pays_sources = {}

for line in sources:
    if re.match(" ", line) or re.match('sources', line, re.IGNORECASE):
        continue
    matches = re.search(r"\(1(?: sources)?(?:\s|\))", line)

    if matches:
        try:
            splitline = line.split(matches.group(0))
            journal = normalise_nom(splitline[0])
            reste = splitline[1].split()
            pays = reste[0]
            liste_pays_sources[journal] = pays
        except IndexError:
            failures.append(line)
    else:
        failures.append(line)

In [8]:
failures

["1775 source(s) trouvées dans l'ensemble de nos solutions. U.I. : Pour usage individuel",
 'Nom de la source PDF Pays Région Type de source Périodicité Langue Org. U.I. B.P. B.E.',
 '93,7Rythme FM (Sherbrooke, QC)(site web réf.) Canada Québec Presse Irrégulier Français',
 '(1 sources)',
 'Actualités Sociales Hebdo - Numéros juridiques France Ile-de-France Presse Bimestriel ou Français',
 "Actualités-L'Étincelle (Windsor, QC) (site web réf.) Canada Québec Presse En continu Français",
 '(1 sources)',
 "Actuel, L' (Haute-St-Charles/Les Rivières/ Canada Québec Presse En continu Français",
 'Wendake, QC) (site web) (1 sources)',
 "Agence Rwandaise d'Information (ARI-RNA)(site Rwanda Presse Irrégulier Français",
 'web réf.) (1 sources)',
 'Autorité des marchés �nanciers (AMF) (site web Canada Québec Presse En continu Français',
 'réf.) (1 sources)',
 "Avenir et Des Rivières, L' (Farnham, QC) (site Canada Québec Presse En continu Français",
 'web) (1 sources)',
 'Bibliothèque et Archives Can

In [14]:
for line in failures:
    input_line = input(line)
    if not input_line:
        continue
    journal = re.split(rf"{re.escape(input_line)}\b", line)[0]
    journal=normalise_nom(journal)
    liste_pays_sources[journal] = input_line


In [16]:
liste_pays_sources

{'01 net': 'France',
 '01 net - Hors-série': 'France',
 'Le 10 Sport (site web réf.)': 'France',
 '107,7 FM Estrie (Sherbrooke, QC) (site web réf.)': 'Canada',
 'Le 18h (site web)': 'France',
 'Les 2 Rives (Sorel-Tracy, QC)': 'Canada',
 'Les 2 Rives (Sorel-Tracy, QC) (site web)': 'Canada',
 '20 Minutes': 'France',
 '20 Minutes (site web)': 'France',
 '24 Heures (Suisse)': 'Suisse',
 '24 heures Montréal': 'Canada',
 '24heureinfo (site web)': 'Togo',
 '45e Nord (site web réf.)': 'Canada',
 '60 millions de consommateurs (site web réf.)': 'France',
 '94 Citoyens (site web réf.)': 'France',
 '98,5 FM (Montréal, QC) (site web réf.)': 'Canada',
 '99 Scenes(site web réf.)': 'Canada',
 'A5 NEWS (site web)': 'Cameroun',
 'ABC Bourse (site web)': 'France',
 'Acadie Nouvelle': 'Canada',
 'Acadie Nouvelle (site web)': 'Canada',
 'Accès Laurentides (QC) (site web réf.)': 'Canada',
 'Acteurs Publics - Nominations (site web)': 'France',
 'Acteurs Publics (site web)': 'France',
 'Action Co': 'France',


In [19]:
with open("liste_pays_sources.json", "w", encoding="utf-8") as f:
    json.dump(liste_pays_sources, f, ensure_ascii=False, indent=4)

In [107]:
with open("liste_pays_sources.json", "r", encoding="utf-8") as f:
    liste_pays_sources = json.load(f)

In [62]:
reader2 = PdfReader("liste_sources_31_mars_2026.pdf")
txt_sources = extract_text_from_pdf(reader2)
with open("sources.txt", "w", encoding="utf-8") as f:
    f.write(txt_sources)

In [140]:
jsons = split_into_articles("pdf_print/2025/11_2025.pdf")

Multiple definitions in dictionary at byte 0x1753f84 for key /x2308
Multiple definitions in dictionary at byte 0x1753f94 for key /x2308
Multiple definitions in dictionary at byte 0x1756473 for key /x2308


In [141]:
merge_json = merge_double(jsons)

In [127]:
liste_pays_lower = {k.lower(): v for k, v in liste_pays_sources.items()}

In [153]:
def assigner_pays(merge_json, liste_pays_sources, liste_pays_lower):
    for json in merge_json:
        for journal in json["journaux"]:
            if journal in liste_pays_sources:
                pays = liste_pays_sources[journal]
            else:
                match = process.extractOne(journal.lower(), liste_pays_lower.keys(), scorer=fuzz.WRatio)
                if match and match[1] >= 90:  # seuil de similarité
                    pays = liste_pays_lower[match[0]]
                elif re.search(r"QC\)", journal) or re.search(r"SRC ", journal):
                    pays = "Canada"
                else:
                    pays = None
            if pays is not None:
                if "pays" not in json:
                    json["pays"] = [pays]
                elif pays not in json["pays"]:
                    json["pays"].append(pays)
    return merge_json

In [147]:
def nom_fichier(article, index=None):
    """
    Génère un nom de fichier à partir des métadonnées du JSON.
    Exemple : le-soleil_12-mars-2024_0042.json
    """
    # Journal de base (sans support, sans localisation)
    journal = article["journaux"][0] if article["journaux"] else "inconnu"
    journal = re.sub(r'\s*\([^)]*\)', '', journal)  # supprime les parenthèses
    journal = journal.strip().lower()
    journal = re.sub(r'[^a-z0-9]+', '-', journal)   # slugify
    if len(journal) > 30:
        journal = journal[:30]

    # Date
    date = article["dates"][0] if article["dates"] else "sans-date"
    date = date.strip().lower()
    date = re.sub(r'\s+', '-', date)
    date = re.sub(r'[^a-z0-9\-]', '', date)

    # Suffixe index pour éviter les collisions
    suffixe = f"_{index:04d}" if index is not None else ""

    return f"{journal}_{date}{suffixe}.json"

In [156]:
def article2txt(file, month, year, index=0):
    jsons = split_into_articles(file)
    merge_json = merge_double(jsons)
    merge_json = assigner_pays(merge_json, liste_pays_sources, liste_pays_lower)
    m_dir = os.path.join(year, month)
    output_folder = os.path.join(OUTPUT_DIR, m_dir)
    os.makedirs(output_folder, exist_ok=True)

    for i, article in enumerate(merge_json):
        if re.search("Ce document référence un lien URL de site non hébergé par Cision.", article["texte"]):
            continue
        filename = nom_fichier(article, index=index + i)
        with open(os.path.join(output_folder, filename), "w", encoding="utf-8") as f:
            json.dump(article, f, ensure_ascii=False, indent=4)
        
    return index

In [167]:
for year in range(2004, 2026) :
    path = os.path.join(SOURCE_DIR, str(year))
    for month in range(1, 13):
        month = str(month) if month > 9 else '0' + str(month)
        file = path + '/' + month + '_' + str(year) + '.pdf'
        if year > 2015:
            if os.path.isfile(file):
                print(f"Ouverture fichier {file}")
                index = article2txt(file, month, str(year))
        else:
            index = len(os.listdir(os.path.join(OUTPUT_DIR, str(year), month))) if os.path.isdir(os.path.join(OUTPUT_DIR, str(year), month)) else 0
        i = 2
        while os.path.isfile(path + '/' + month + '_' + str(year) + '_' + str(i) + '.pdf'):
            print(f"Ouverture fichier {path + '/' + month + '_' + str(year) + '_' + str(i) + '.pdf'}")
            index = article2txt(file, month, str(year), index=index)
            i = i + 1


Ouverture fichier pdf_print/2006/12_2006_2.pdf
Ouverture fichier pdf_print/2010/11_2010_2.pdf
Ouverture fichier pdf_print/2011/01_2011_2.pdf
Ouverture fichier pdf_print/2011/02_2011_2.pdf
Ouverture fichier pdf_print/2011/03_2011_2.pdf
Ouverture fichier pdf_print/2011/04_2011_2.pdf
Ouverture fichier pdf_print/2011/04_2011_3.pdf
Ouverture fichier pdf_print/2011/05_2011_2.pdf
Ouverture fichier pdf_print/2011/06_2011_2.pdf
Ouverture fichier pdf_print/2011/07_2011_2.pdf
Ouverture fichier pdf_print/2011/09_2011_2.pdf
Ouverture fichier pdf_print/2011/10_2011_2.pdf
Ouverture fichier pdf_print/2011/10_2011_3.pdf
Ouverture fichier pdf_print/2011/11_2011_2.pdf
Ouverture fichier pdf_print/2011/12_2011_2.pdf
Ouverture fichier pdf_print/2012/01_2012_2.pdf
Ouverture fichier pdf_print/2012/02_2012_2.pdf
Ouverture fichier pdf_print/2012/03_2012_2.pdf
Ouverture fichier pdf_print/2012/04_2012_2.pdf
Ouverture fichier pdf_print/2012/05_2012_2.pdf
Ouverture fichier pdf_print/2012/06_2012_2.pdf
Ouverture fic

Multiple definitions in dictionary at byte 0x1933158 for key /x7157
Multiple definitions in dictionary at byte 0x1933168 for key /x7157


Ouverture fichier pdf_print/2025/09_2025_2.pdf


Multiple definitions in dictionary at byte 0x1933158 for key /x7157
Multiple definitions in dictionary at byte 0x1933168 for key /x7157


Ouverture fichier pdf_print/2025/10_2025.pdf


...

Ouverture fichier pdf_print/2025/11_2025.pdf


Multiple definitions in dictionary at byte 0x1753f84 for key /x2308
Multiple definitions in dictionary at byte 0x1753f94 for key /x2308
Multiple definitions in dictionary at byte 0x1756473 for key /x2308


Ouverture fichier pdf_print/2025/11_2025_2.pdf


Multiple definitions in dictionary at byte 0x1753f84 for key /x2308
Multiple definitions in dictionary at byte 0x1753f94 for key /x2308
Multiple definitions in dictionary at byte 0x1756473 for key /x2308


Ouverture fichier pdf_print/2025/12_2025.pdf
Ouverture fichier pdf_print/2025/12_2025_2.pdf
Ouverture fichier pdf_print/2025/12_2025_3.pdf


In [143]:
merge_json

[{'journaux': ['France Inter (site web réf.) - France Inter',
   'France Culture (site web réf.)'],
  'dates': ['14 novembre 2025', '13 novembre 2025'],
  'texte': '"Kika" et ses sœurs prostituées\nCette semaine sort en salles le premier film de fiction de la réalisatrice Alexe Poukine. Un portrait naturaliste d\'une jeune femme endeuillée\net endettée qui a recours à la prostitution... Voir l\'article\nCe document référence un lien URL de site non hébergé par Cision.\nLe présent document est protégé par les lois et conventions internationales sur le droit d\'auteur et son utilisation est régie par ces lois et conventions.\n web·20251114·WPAAA·1158863_10336371431_11727151\nLe Courrier de l\'Ouest\nAngers\nAngers, mercredi 12 novembre 2025 784 mots, p. Le Courrier de l\'Ouest Angers_6\nAussi paru dans 11 novembre 2025 -Le Courrier de l\'Ouest (site web)\n« Une intimité au-delà du sexuel »\nLELIAN\nAlexe Poukine est venue présenter en avant-première à Angers, au public des 400 Coups, son

In [5]:
random.seed(26112002)
os.makedirs("random_sample", exist_ok=True)
for folder in os.listdir(OUTPUT_DIR):
    for month in os.listdir(os.path.join(OUTPUT_DIR, folder)):
        files = os.listdir(os.path.join(OUTPUT_DIR, folder, month))
        random_file = random.sample(files, 1)
        with open(os.path.join(OUTPUT_DIR, folder, month, random_file[0]), "r", encoding="utf-8") as f:
            article = json.load(f)
        filename = os.path.splitext(random_file[0])[0] + ".txt"
        with open(os.path.join("random_sample", filename), "w", encoding="utf-8") as f:
            f.write(article["texte"])




In [5]:
tasks = []
for file in os.listdir("random_sample"):
    with open(os.path.join("random_sample", file), "r", encoding="utf-8") as f:
        texte = f.read()
    
    task = {"text": texte}
    tasks.append(task)

  # Affiche les 500 premiers caractères de chaque article

In [7]:
resp = client.projects.import_tasks(

    id=2,

    request=tasks,

    return_task_ids=True,

)

print(resp)

annotation_count=0 could_be_tasks_list=False data_columns=[] duration=0.09724545478820801 file_upload_ids=[] found_formats=[] import_=None predictions_count=None task_count=312 prediction_count=0 task_ids=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

# Dataset d'annotaion pour le NER (Vérité de terrain)

In [2]:
gt_path = "GT/data_annotee_sujet.json"
with open(gt_path, "r") as f:
    gt_data = json.load(f)

In [8]:
marginal = []
pertinent = []

marginal_fiction = 0
for task in gt_data:
    for annotation in task["annotations"]:
        for result in annotation["result"]:
            if "choices" in result.get("value", {}):
                if "Mention marginale" in result["value"]["choices"]:
                    marginal.append(task["data"]["text"])
                else:
                    pertinent.append(task["data"]["text"])



In [10]:
while "" in marginal:    marginal.remove("")
while "" in pertinent:    pertinent.remove("")

n_pertinent = 15
n_marginal = 5

random.seed(26112002)

# Vérifications
if len(pertinent) < n_pertinent:
    raise ValueError(f"La liste 'pertinent' contient moins de {n_pertinent} textes.")
if len(marginal) < n_marginal:
    raise ValueError(f"La liste 'marginal' contient moins de {n_marginal} textes.")

# Échantillonnage aléatoire sans remise
sample_pertinent = random.sample(pertinent, n_pertinent)
sample_marginal = random.sample(marginal, n_marginal)

# Construction des tâches Label Studio
tasks = []

for i, texte in enumerate(sample_pertinent):
    tasks.append({
        "id": f"pertinent_{i}",
        "data": {
            "text": texte,
            "source": "pertinent"
        }
    })

for i, texte in enumerate(sample_marginal):
    tasks.append({
        "id": f"marginal_{i}",
        "data": {
            "text": texte,
            "source": "marginal"
        }
    })

# Mélanger l'ordre final si vous voulez éviter les blocs par classe
random.shuffle(tasks)

# Sauvegarde en JSON
with open("labelstudio_import.json", "w", encoding="utf-8") as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)

## Dataset d'annotation via LLM

In [2]:
import polars as pl

In [4]:
sample = (
    pl.scan_parquet("corpus/corpus.parquet")
      .select(["id", "texte"])
      .collect()
      .sample(n=500, seed=26112002)
)

In [ ]:
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
import anthropic
import os

In [18]:
import time
import re
import json

In [13]:
client = anthropic.Client()

In [11]:
prompt_system = """Tu es un annotateur NLP.
Tâche : extraire dans le texte les segments correspondant aux catégories du schéma.
Règles :
- Respect strict des définitions.
- Extraire uniquement des segments exacts du texte (pas de reformulation).
- CLIENT, PROX, PROST, CONCEPT_PROST = noms communs, syntagmes nominaux ; ACT_PROST = formes ou syntagme verbaux.
- Ignore les quantifieurs, numéraux, articles, démonstratifs ou adjectifs descriptifs non essentiels.
- Ignore les groupes nominaux trop généraux ou trop larges si le noyau ne correspond pas clairement à une catégorie du schéma.
- Pas de noms propres, pronoms, ni inférences.
- Extraire toutes les occurrences (doublons autorisés).
- Ne pas confondre personne / action / concept.
Sortie :
- JSON uniquement, format :
{
  "CLIENT": [],
  "PROX": [],
  "PROST": [],
  "ACT_PROST": [],
  "CONCEPT_PROST": []
}
- Listes vides si rien.
Schéma :
schema = {
    "CLIENT": "personne achetant acte sexuel",
    "PROX": "personne organisant/profitant prostitution",
    "PROST": "personne fournissant actes sexuels rémunérés",
    "ACT_PROST": "action de prostitution ou sollicitation",
    "CONCEPT_PROST": "concept/système lié à prostitution"
}"""

In [12]:
request_list = []
for id_, text in sample.iter_rows():
    request = Request(
            custom_id=id_,
            params=MessageCreateParamsNonStreaming(
                model="claude-sonnet-4-6",
                max_tokens=1024,
                system=prompt_system,
                messages=[
                    {
                        "role": "user",
                        "content": f"texte: {text}",
                    }
                ],
            ),
        )
    
    request_list.append(request)


In [15]:
message_batch = client.messages.batches.create(requests=request_list)
batch_id = message_batch.id

message_batch_info = None
while True:
    message_batch_info = client.messages.batches.retrieve(batch_id)
    if message_batch_info.processing_status == "ended":
        break

    print(f"Batch {batch_id} is still processing...")
    time.sleep(60)
print(message_batch_info)



Batch msgbatch_01XNryR4Kz7Wt2SYV3fhSmNK is still processing...
Batch msgbatch_01XNryR4Kz7Wt2SYV3fhSmNK is still processing...
Batch msgbatch_01XNryR4Kz7Wt2SYV3fhSmNK is still processing...
Batch msgbatch_01XNryR4Kz7Wt2SYV3fhSmNK is still processing...
MessageBatch(id='msgbatch_01XNryR4Kz7Wt2SYV3fhSmNK', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 29, 12, 1, 50, 172436, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 4, 29, 12, 5, 43, 51491, tzinfo=TzInfo(0)), expires_at=datetime.datetime(2026, 4, 30, 12, 1, 50, 172436, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=500), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01XNryR4Kz7Wt2SYV3fhSmNK/results', type='message_batch')


In [17]:
def parse_batch_item(item):
    text = item.result.message.content[0].text
    m = re.search(r"```json\s*(.*?)\s*```", text, re.S)
    json_str = m.group(1) if m else text
    data = json.loads(json_str)
    return item.custom_id, data

In [19]:
result_dict_claude = {}
for result in client.messages.batches.results(batch_id):
    try:
        item, data = parse_batch_item(result)
        result_dict_claude[item] = data
    except Exception as e:
        print(f"Error parsing result: {e}")
        print(f"Raw content: {result.result.message.content[0].text}")

In [21]:
def make_chunk_sentence(text, n=10):
    sentences = text.split('.')
    tmp = []
    chunks = []
    counter = 0
    if n <= 0:
        return
    for sentence in sentences:
        if counter < n:
            tmp.append(sentence)
            counter += 1
        else:
            chunk = '. '.join(tmp) +'.'
            chunks.append(chunk)
            tmp = [sentence]
            counter = 1
    
    if tmp:
        chunk = '. '.join(tmp)+'.'
        chunks.append(chunk)
    
    return chunks

In [25]:
def make_pattern(term):
    term_escaped = re.escape(term)
    if " " in term:
        return term_escaped
    return r"\b" + term_escaped + r"\b"

def extract_spans(text, entities):
    results = []
    for label, terms in entities.items():
        for term in set(terms):
            pattern = make_pattern(term)
            for m in re.finditer(pattern, text, flags=re.IGNORECASE):
                results.append({
                    "start": m.start(),
                    "end": m.end(),
                    "text": m.group(0),
                    "labels": [label]
                })
    return sorted(results, key=lambda x: (x["start"], x["end"]))

In [31]:
preds = []
counter = 0
for id_, text in sample.iter_rows():
    chunks = make_chunk_sentence(text, n=2)
    pred_list = [val for liste_valeurs in result_dict_claude[id_].values() for val in liste_valeurs]
    for chunk in chunks:
        for pred in pred_list:
            if pred in chunk:
                chunk = re.sub(r"\n+", " ", chunk)
                counter += 1
                labels = extract_spans(chunk, result_dict_claude[id_])
                dict_tmp = {"id": counter,
                "label": {
                    "text": chunk,
                    "labels": labels
                }}
                preds.append(dict_tmp)
                break



In [32]:
with open("predictions/preds_NER_LLM.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)

In [ ]:
def convert_to_labelstudio(input_json):
    converted_tasks = []
    for item in input_json:
        task = {
            "data": {"label": {"text": item["label"]["text"]}},
            "annotations": [{
                "result": []
            }]
        }
        for lbl in item["label"].get("labels", []):
            result = {
                "from_name": "label", 
                "to_name": "text",    
                "type": "labels",
                "value": {
                    "start": lbl["start"],
                    "end": lbl["end"],
                    "text": lbl["text"],
                    "labels": lbl["labels"]
                }
            }
            task["annotations"][0]["result"].append(result)
        converted_tasks.append(task)
    return converted_tasks

# Usage :
new_data = convert_to_labelstudio(preds)
with open('predictions/NER_LLM_labelstudio.json', 'w', encoding='utf-8') as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)